# Table Of Contents

- [Reading a csv file](#Reading-from-a-csv-file)
- [Quick Summary Statistics](#Quick-Summary-Statistics)
- [Sorting Columns](#Sorting-Columns)
- [Removing duplicates](#Removing-duplicates)
- [Finding duplicates](#Finding-duplicates)
- [Creating a new column based on values in other columns](#Creating-a-new-column-based-on-values-in-other-columns)
- [Working with dates](#Working-with-dates)
- [Creating row number column](#Creating-row-number-column)
- [Creating rank column](#Creating-rank-column)
- [Forward fill null values](#Forward-fill-null-values)
- [Inject arbitrary string into new column](#Inject-arbitrary-string-into-new-column)

## Reading from a csv file

[Back to Top](#Table-Of-Contents)

#### Reading from local CSV file

In [1]:
!head -n 5 cuts.csv

ï»¿Year,Month,Month_Number,Data_Item,Value,Units,Source
2024,August,8,Ground chuck retail price,5.638,Dollars per pound,BLS
2024,September,9,Ground chuck retail price,5.576,Dollars per pound,BLS
2024,October,10,Ground chuck retail price,5.586,Dollars per pound,BLS
2024,November,11,Ground chuck retail price,5.605,Dollars per pound,BLS


In [2]:
import polars as pl

df = pl.read_csv(
    "cuts.csv"            
)

In [3]:
df.head()

Year,Month,Month_Number,Data_Item,Value,Units,Source
i64,str,i64,str,str,str,str
2024,"""August""",8,"""Ground chuck retail price""","""5.638""","""Dollars per pound""","""BLS"""
2024,"""September""",9,"""Ground chuck retail price""","""5.576""","""Dollars per pound""","""BLS"""
2024,"""October""",10,"""Ground chuck retail price""","""5.586""","""Dollars per pound""","""BLS"""
2024,"""November""",11,"""Ground chuck retail price""","""5.605""","""Dollars per pound""","""BLS"""
2024,"""December""",12,"""Ground chuck retail price""","""5.576""","""Dollars per pound""","""BLS"""


#### Reading from remote URL if you get a 403 Forbidden Error

In [4]:
import io
import requests
import polars as pl

url = "https://download.bls.gov/pub/time.series/cu/cu.series"
headers = {"User-Agent": "Your Name your.email@example.com"}

resp = requests.get(url, headers=headers)
resp.raise_for_status()

df = pl.read_csv(
    io.BytesIO(resp.content),
    separator="\t",
    infer_schema_length=10000
)

# Remove whitespace around column names
df = df.rename({c: c.strip() for c in df.columns})

# Remove whitespadce around column values
df = df.with_columns(pl.col(pl.String).str.strip_chars())

## Set the schema/data type of each column

[Back to Top](#Table-Of-Contents)

`schema={key: value}` is for ALL colunms, for a few columns, use `schema_overrides={key: value}`

In [5]:
df = pl.read_csv(
    "cuts.csv",
    schema={
        "Year": pl.Int32,
        "Month": pl.String,
        "Month_Number": pl.Int32,
        "Data_Item": pl.String,
        "Value": pl.Float32,
        "Units": pl.String,
        "Source": pl.String
    },
    null_values="NA",
)

In [6]:
df.head()

Year,Month,Month_Number,Data_Item,Value,Units,Source
i32,str,i32,str,f32,str,str
2024,"""August""",8,"""Ground chuck retail price""",5.638,"""Dollars per pound""","""BLS"""
2024,"""September""",9,"""Ground chuck retail price""",5.576,"""Dollars per pound""","""BLS"""
2024,"""October""",10,"""Ground chuck retail price""",5.586,"""Dollars per pound""","""BLS"""
2024,"""November""",11,"""Ground chuck retail price""",5.605,"""Dollars per pound""","""BLS"""
2024,"""December""",12,"""Ground chuck retail price""",5.576,"""Dollars per pound""","""BLS"""


## Quick Summary Statistics

[Back to Top](#Table-Of-Contents)

In [7]:
df.shape    # (rows, columns), e.g. (1000, 8)

(800, 7)

In [8]:
df.height   # number of rows

800

In [9]:
len(df)     # also the number of rows

800

In [10]:
df.width    # number of columns

7

In [11]:
df.schema

Schema([('Year', Int32),
        ('Month', String),
        ('Month_Number', Int32),
        ('Data_Item', String),
        ('Value', Float32),
        ('Units', String),
        ('Source', String)])

In [12]:
df.columns

['Year', 'Month', 'Month_Number', 'Data_Item', 'Value', 'Units', 'Source']

In [13]:
df.null_count()

Year,Month,Month_Number,Data_Item,Value,Units,Source
u32,u32,u32,u32,u32,u32,u32
0,0,0,0,31,0,0


In [14]:
df.glimpse()

Rows: 800
Columns: 7
$ Year         <i32> 2024, 2024, 2024, 2024, 2024, 2025, 2025, 2025, 2025, 2025
$ Month        <str> 'August', 'September', 'October', 'November', 'December', 'January', 'February', 'March', 'April', 'May'
$ Month_Number <i32> 8, 9, 10, 11, 12, 1, 2, 3, 4, 5
$ Data_Item    <str> 'Ground chuck retail price', 'Ground chuck retail price', 'Ground chuck retail price', 'Ground chuck retail price', 'Ground chuck retail price', 'Ground chuck retail price', 'Ground chuck retail price', 'Ground chuck retail price', 'Ground chuck retail price', 'Ground chuck retail price'
$ Value        <f32> 5.638000011444092, 5.576000213623047, 5.585999965667725, 5.605000019073486, 5.576000213623047, 5.494999885559082, 5.74399995803833, 5.854000091552734, 5.995999813079834, 6.01800012588501
$ Units        <str> 'Dollars per pound', 'Dollars per pound', 'Dollars per pound', 'Dollars per pound', 'Dollars per pound', 'Dollars per pound', 'Dollars per pound', 'Dollars per pound', 'Dollars per 

## Sorting Columns

[Back to Top](#Table-Of-Contents)

#### Sorting one column

In [15]:
df.sort("Month", descending=True)

Year,Month,Month_Number,Data_Item,Value,Units,Source
i32,str,i32,str,f32,str,str
2024,"""September""",9,"""Ground chuck retail price""",5.576,"""Dollars per pound""","""BLS"""
2025,"""September""",9,"""Ground chuck retail price""",6.326,"""Dollars per pound""","""BLS"""
2024,"""September""",9,"""Ground beef retail price""",5.67,"""Dollars per pound""","""BLS"""
2025,"""September""",9,"""Ground beef retail price""",6.323,"""Dollars per pound""","""BLS"""
2024,"""September""",9,"""Lean & extra lean ground beef …",7.223,"""Dollars per pound""","""BLS"""
…,…,…,…,…,…,…
2026,"""April""",4,"""American processed cheese reta…",4.69,"""Dollars per pound""","""BLS"""
2025,"""April""",4,"""Cheddar cheese, natural retail…",5.743,"""Dollars per pound""","""BLS"""
2026,"""April""",4,"""Cheddar cheese, natural retail…",6.029,"""Dollars per pound""","""BLS"""


#### Sorting multiple columns

In [16]:
df.sort(["Year","Month_Number"], descending=[False, True])

Year,Month,Month_Number,Data_Item,Value,Units,Source
i32,str,i32,str,f32,str,str
2024,"""December""",12,"""Eggs, Grade A large retail pri…",4.146,"""Dollars per dozen""","""BLS"""
2024,"""December""",12,"""All other pork excluding canne…",3.656,"""Dollars per pound""","""BLS"""
2024,"""December""",12,"""Retail broiler composite""",243.399994,"""Cents per pound""","""ERS"""
2024,"""December""",12,"""All ham (not canned or sliced)…",4.406,"""Dollars per pound""","""BLS"""
2024,"""December""",12,"""Ground chuck retail price""",5.576,"""Dollars per pound""","""BLS"""
…,…,…,…,…,…,…
2026,"""January""",1,"""Pork chops, center cut, bone i…",4.453,"""Dollars per pound""","""BLS"""
2026,"""January""",1,"""Wholesale broiler composite""",85.699997,"""Cents per pound""","""ERS"""
2026,"""January""",1,"""Beef for stew, boneless retail…",8.463,"""Dollars per pound""","""BLS"""


## Removing duplicates

[Back to Top](#Table-Of-Contents)

In [17]:
df = pl.DataFrame(
    {
        'k1': ['one'] * 3 + ['two'] * 4,
         'k2': [3, 2, 1, 3, 3, 4, 4]
    },
    schema={
        "k1": pl.String,
        "k2": pl.Int32,
    }
)

In [18]:
df

k1,k2
str,i32
"""one""",3
"""one""",2
"""one""",1
"""two""",3
"""two""",3
"""two""",4
"""two""",4


### Removing duplicates within one column

In [19]:
df.unique(subset="k1")

k1,k2
str,i32
"""one""",3
"""two""",3


### Removing duplicates within multiple columns

In [20]:
df.unique(subset=["k1","k2"])

k1,k2
str,i32
"""two""",3
"""one""",3
"""two""",4
"""one""",1
"""one""",2


## Finding duplicates

[Back to Top](#Table-Of-Contents)

In [21]:
df.filter(pl.struct(["k1", "k2"]).is_duplicated())

k1,k2
str,i32
"""two""",3
"""two""",3
"""two""",4
"""two""",4


But if you just want the first unique values of the duplicates:

In [22]:
df.filter(~pl.struct(["k1", "k2"]).is_first_distinct())

k1,k2
str,i32
"""two""",3
"""two""",4


## Creating a new column based on values in other columns

[Back to Top](#Table-Of-Contents)

In [23]:
df = pl.DataFrame(
    {
        'food': ['bacon', 'pulled pork', 'bacon', 'pastrami','corned beef', 'bacon', 'pastrami', 'honey ham','nova lox'],
        'ounces': [4, 3, 12, 6, 7.5, 8, 3, 5, 6]
    },
    schema={
        "food": pl.String,
        "ounces": pl.Float32
    }
)

In [24]:
df

food,ounces
str,f32
"""bacon""",4.0
"""pulled pork""",3.0
"""bacon""",12.0
"""pastrami""",6.0
"""corned beef""",7.5
"""bacon""",8.0
"""pastrami""",3.0
"""honey ham""",5.0
"""nova lox""",6.0


**Using a mapping dictionary**

In [25]:
mapping = {
    "bacon": "pig",
    "honey ham": "pig",
    "pulled pork": "pig",
    "pastrami": "cow",
    "corned beef": "cow",
    "nova lox": "salmon",
}

df = df.with_columns(
    pl.col("food").replace_strict(mapping).alias("animal")
)

In [26]:
df

food,ounces,animal
str,f32,str
"""bacon""",4.0,"""pig"""
"""pulled pork""",3.0,"""pig"""
"""bacon""",12.0,"""pig"""
"""pastrami""",6.0,"""cow"""
"""corned beef""",7.5,"""cow"""
"""bacon""",8.0,"""pig"""
"""pastrami""",3.0,"""cow"""
"""honey ham""",5.0,"""pig"""
"""nova lox""",6.0,"""salmon"""


**Using combination of with_columns() and when().then() pattern**

In [27]:
df = df.with_columns(
    pl.when(pl.col("food").is_in(["bacon", "honey ham", "pulled pork"])).then(pl.lit("pig"))
      .when(pl.col("food").is_in(["pastrami", "corned beef"])).then(pl.lit("cow"))
      .when(pl.col("food") == "nova lox").then(pl.lit("salmon"))
      .otherwise(pl.lit(None))
      .alias("animal2")
).with_columns(
    pl.when((pl.col("animal") == "pig") & (pl.col("ounces") > 4)).then(pl.lit("Big Pig"))
      .when((pl.col("animal") == "pig") & (pl.col("ounces") < 4)).then(pl.lit("Small Pig"))
      .otherwise(pl.lit(None))
      .alias("animal_type")
)

In [28]:
df

food,ounces,animal,animal2,animal_type
str,f32,str,str,str
"""bacon""",4.0,"""pig""","""pig""",null
"""pulled pork""",3.0,"""pig""","""pig""","""Small Pig"""
"""bacon""",12.0,"""pig""","""pig""","""Big Pig"""
"""pastrami""",6.0,"""cow""","""cow""",null
"""corned beef""",7.5,"""cow""","""cow""",null
"""bacon""",8.0,"""pig""","""pig""","""Big Pig"""
"""pastrami""",3.0,"""cow""","""cow""",null
"""honey ham""",5.0,"""pig""","""pig""","""Big Pig"""
"""nova lox""",6.0,"""salmon""","""salmon""",null


## Working with dates

[Back to Top](#Table-Of-Contents)

In [29]:
import numpy as np
import polars as pl
from datetime import date

dates = pl.date_range(date(2013, 1, 1), date(2013, 1, 10), interval="1d", eager=True)

df = pl.DataFrame({
    "value": np.random.randn(10),
    "date": dates,
})
df

value,date
f64,date
1.44712,2013-01-01
-0.461834,2013-01-02
-0.327243,2013-01-03
-1.086488,2013-01-04
0.275612,2013-01-05
0.371901,2013-01-06
-0.9367,2013-01-07
-1.806295,2013-01-08
1.895856,2013-01-09


**Creating string YYYY-MM values**

In [30]:
df = df.with_columns(
    year_month=pl.col("date").dt.strftime("%Y-%m")
)
df

value,date,year_month
f64,date,str
1.44712,2013-01-01,"""2013-01"""
-0.461834,2013-01-02,"""2013-01"""
-0.327243,2013-01-03,"""2013-01"""
-1.086488,2013-01-04,"""2013-01"""
0.275612,2013-01-05,"""2013-01"""
0.371901,2013-01-06,"""2013-01"""
-0.9367,2013-01-07,"""2013-01"""
-1.806295,2013-01-08,"""2013-01"""
1.895856,2013-01-09,"""2013-01"""


**Creating separate year and month values as integers**

In [31]:
df = df.with_columns(
    year=pl.col("date").dt.year(),
    month=pl.col("date").dt.month(),
)
df

value,date,year_month,year,month
f64,date,str,i32,i8
1.44712,2013-01-01,"""2013-01""",2013,1
-0.461834,2013-01-02,"""2013-01""",2013,1
-0.327243,2013-01-03,"""2013-01""",2013,1
-1.086488,2013-01-04,"""2013-01""",2013,1
0.275612,2013-01-05,"""2013-01""",2013,1
0.371901,2013-01-06,"""2013-01""",2013,1
-0.9367,2013-01-07,"""2013-01""",2013,1
-1.806295,2013-01-08,"""2013-01""",2013,1
1.895856,2013-01-09,"""2013-01""",2013,1


**Creating an actual date**

In [32]:
df = df.with_columns(
    month_start=pl.col("date").dt.truncate("1mo")
)
df

value,date,year_month,year,month,month_start
f64,date,str,i32,i8,date
1.44712,2013-01-01,"""2013-01""",2013,1,2013-01-01
-0.461834,2013-01-02,"""2013-01""",2013,1,2013-01-01
-0.327243,2013-01-03,"""2013-01""",2013,1,2013-01-01
-1.086488,2013-01-04,"""2013-01""",2013,1,2013-01-01
0.275612,2013-01-05,"""2013-01""",2013,1,2013-01-01
0.371901,2013-01-06,"""2013-01""",2013,1,2013-01-01
-0.9367,2013-01-07,"""2013-01""",2013,1,2013-01-01
-1.806295,2013-01-08,"""2013-01""",2013,1,2013-01-01
1.895856,2013-01-09,"""2013-01""",2013,1,2013-01-01


## Creating row number column

[Back to Top](#Table-Of-Contents)

In [33]:
df = pl.read_csv("row_num.csv").drop("ROW_NUM")
df

CLAIM_NUM,PART_NUM,PART_COST_USD,LABOR_COST_USD,HANDLING_COST_USD,TOTAL_COST_USD
i64,str,f64,f64,f64,f64
1,"""062015LH""",303.13,80.45,35.34,751.97
1,"""062315LH""",645.33,60.34,46.3,751.97
1,"""062345LH""",323.55,67.25,20.56,751.97
2,"""062015LH""",300.25,80.45,35.34,720.09
2,"""062315LH""",613.45,60.34,46.3,720.09
3,"""062345LH""",333.1,67.25,20.56,420.91
4,"""062345LH""",300.25,80.45,46.3,427.0


In [34]:
df = df.with_columns(
    pl.col("CLAIM_NUM").cum_count().over("CLAIM_NUM").alias("ROW_NUM")
)
df

CLAIM_NUM,PART_NUM,PART_COST_USD,LABOR_COST_USD,HANDLING_COST_USD,TOTAL_COST_USD,ROW_NUM
i64,str,f64,f64,f64,f64,u32
1,"""062015LH""",303.13,80.45,35.34,751.97,1
1,"""062315LH""",645.33,60.34,46.3,751.97,2
1,"""062345LH""",323.55,67.25,20.56,751.97,3
2,"""062015LH""",300.25,80.45,35.34,720.09,1
2,"""062315LH""",613.45,60.34,46.3,720.09,2
3,"""062345LH""",333.1,67.25,20.56,420.91,1
4,"""062345LH""",300.25,80.45,46.3,427.0,1


**If you need row numbering to be based on sort order of a certain column**

In [35]:
df = df.with_columns(
    pl.col("CLAIM_NUM").cum_count().over("CLAIM_NUM", order_by="PART_NUM").alias("ROW_NUM")
)
df

CLAIM_NUM,PART_NUM,PART_COST_USD,LABOR_COST_USD,HANDLING_COST_USD,TOTAL_COST_USD,ROW_NUM
i64,str,f64,f64,f64,f64,u32
1,"""062015LH""",303.13,80.45,35.34,751.97,1
1,"""062315LH""",645.33,60.34,46.3,751.97,2
1,"""062345LH""",323.55,67.25,20.56,751.97,3
2,"""062015LH""",300.25,80.45,35.34,720.09,1
2,"""062315LH""",613.45,60.34,46.3,720.09,2
3,"""062345LH""",333.1,67.25,20.56,420.91,1
4,"""062345LH""",300.25,80.45,46.3,427.0,1


In [36]:
df.filter(pl.col("ROW_NUM")==1)

CLAIM_NUM,PART_NUM,PART_COST_USD,LABOR_COST_USD,HANDLING_COST_USD,TOTAL_COST_USD,ROW_NUM
i64,str,f64,f64,f64,f64,u32
1,"""062015LH""",303.13,80.45,35.34,751.97,1
2,"""062015LH""",300.25,80.45,35.34,720.09,1
3,"""062345LH""",333.1,67.25,20.56,420.91,1
4,"""062345LH""",300.25,80.45,46.3,427.0,1


In [37]:
df = pl.read_csv("scores.csv")
df

SUBJECT,SCORE,STUDENT
str,i64,str
"""Math""",94,"""Jack"""
"""English""",85,"""Julie"""
"""Biology""",98,"""Sam"""
"""Biology""",92,"""Daniel"""
"""Math""",89,"""Robert"""
…,…,…
"""Science""",83,"""David"""
"""Math""",79,"""Sammy"""
"""Biology""",78,"""Alex"""


In [38]:
df = df.with_columns(
    pl.col("SUBJECT").cum_count().over("SUBJECT", order_by="SCORE", descending=True).alias("ROW_NUM")
).sort(["SUBJECT","ROW_NUM"])
df

SUBJECT,SCORE,STUDENT,ROW_NUM
str,i64,str,u32
"""Biology""",98,"""Sam""",1
"""Biology""",92,"""Daniel""",2
"""Biology""",92,"""Jim""",3
"""Biology""",78,"""Alex""",4
"""English""",91,"""Andy""",1
…,…,…,…
"""Math""",94,"""Jack""",1
"""Math""",89,"""Robert""",2
"""Math""",79,"""Sammy""",3


## Creating rank column

[Back to Top](#Table-Of-Contents)

`method=` determines how ties are handled: min, max, ordinal, or dense

In [39]:
df = pl.read_csv("scores.csv")

df = df.with_columns(
    pl.col("SCORE")
    .rank(method="dense", descending=True)
    .over("SUBJECT")
    .alias("RANK")
).sort(["SUBJECT","RANK"])
df

SUBJECT,SCORE,STUDENT,RANK
str,i64,str,u32
"""Biology""",98,"""Sam""",1
"""Biology""",92,"""Daniel""",2
"""Biology""",92,"""Jim""",2
"""Biology""",78,"""Alex""",3
"""English""",91,"""Andy""",1
…,…,…,…
"""Math""",94,"""Jack""",1
"""Math""",89,"""Robert""",2
"""Math""",79,"""Sammy""",3


## Forward fill null values

[Back to Top](#Table-Of-Contents)

In [40]:
df = pl.DataFrame({
    "GROUP": ["A", "A", "A", "B", "B", "B"],
    "VALUE": [1, None, None, None, 5, None],
    "PRICE": [10.0, None, 12.5, None, None, 20.0],
})
df

GROUP,VALUE,PRICE
str,i64,f64
"""A""",1,10.0
"""A""",null,null
"""A""",null,12.5
"""B""",null,null
"""B""",5,null
"""B""",null,20.0


In [41]:
df.with_columns(
    pl.col("VALUE").fill_null(strategy="forward")
)

GROUP,VALUE,PRICE
str,i64,f64
"""A""",1,10.0
"""A""",1,null
"""A""",1,12.5
"""B""",1,null
"""B""",5,null
"""B""",5,20.0


In [42]:
df.with_columns(
    pl.col(["VALUE","PRICE"]).fill_null(strategy="forward")
)

GROUP,VALUE,PRICE
str,i64,f64
"""A""",1,10.0
"""A""",1,10.0
"""A""",1,12.5
"""B""",1,12.5
"""B""",5,12.5
"""B""",5,20.0


**Forward fill within groups to prevent a value from one group leaking into another group**

In [43]:
df.with_columns(
    pl.col(["VALUE","PRICE"]).fill_null(strategy="forward").over("GROUP")
)

GROUP,VALUE,PRICE
str,i64,f64
"""A""",1,10.0
"""A""",1,10.0
"""A""",1,12.5
"""B""",null,null
"""B""",5,null
"""B""",5,20.0


Can also use `forward_fill()` short-hand method

In [44]:
df.with_columns(
    pl.col("VALUE").forward_fill()
)

GROUP,VALUE,PRICE
str,i64,f64
"""A""",1,10.0
"""A""",1,null
"""A""",1,12.5
"""B""",1,null
"""B""",5,null
"""B""",5,20.0


In [45]:
df.with_columns(
    pl.col(["VALUE","PRICE"]).forward_fill()
)

GROUP,VALUE,PRICE
str,i64,f64
"""A""",1,10.0
"""A""",1,10.0
"""A""",1,12.5
"""B""",1,12.5
"""B""",5,12.5
"""B""",5,20.0


## Creating cumulative sum column

[Back to Top](#Table-Of-Contents)

In [46]:
df = pl.read_csv("scores.csv")

df = df.with_columns(
    pl.col("SCORE").cum_sum().over("SUBJECT").alias("CUM_SUM")
).sort(["SUBJECT","CUM_SUM"])
df

SUBJECT,SCORE,STUDENT,CUM_SUM
str,i64,str,i64
"""Biology""",98,"""Sam""",98
"""Biology""",92,"""Daniel""",190
"""Biology""",78,"""Alex""",268
"""Biology""",92,"""Jim""",360
"""English""",85,"""Julie""",85
…,…,…,…
"""Math""",94,"""Jack""",94
"""Math""",89,"""Robert""",183
"""Math""",79,"""Sammy""",262


## Inject arbitrary string into new column

[Back to Top](#Table-Of-Contents)

In [47]:
df = pl.read_csv("scores.csv")

In [48]:
df = (
    df
    .with_columns(pl.lit("Injected String").alias("new_column"))
)
df

SUBJECT,SCORE,STUDENT,new_column
str,i64,str,str
"""Math""",94,"""Jack""","""Injected String"""
"""English""",85,"""Julie""","""Injected String"""
"""Biology""",98,"""Sam""","""Injected String"""
"""Biology""",92,"""Daniel""","""Injected String"""
"""Math""",89,"""Robert""","""Injected String"""
…,…,…,…
"""Science""",83,"""David""","""Injected String"""
"""Math""",79,"""Sammy""","""Injected String"""
"""Biology""",78,"""Alex""","""Injected String"""
